# 🎬 SentimentSage AI — Full Training Notebook

| **Project** | SentimentSage AI — Movie Review Sentiment Analyzer |
|---|---|
| **Dataset** | IMDB 50K Movie Reviews (Kaggle) |
| **Models** | LSTM (trained from scratch) + BERT (HuggingFace inference) |
| **RAG** | Cosine similarity on LSTM hidden state vectors |
| **DAG** | 8-node Directed Acyclic Graph pipeline |
| **Training Time** | ~15-20 min on Colab T4 GPU |
| **Outputs** | lstm_model.pt, tokenizer.pkl, rag_store.pkl, vocab.pkl |

---
**KEY:** You cleaned the data, you built the vocab, you trained the LSTM, you built the RAG store — you can explain every single decision. That is what impresses a teacher.

## Section 1 — Install & Import

In [ ]:
# Install dependencies
!pip install kaggle transformers torch torchtext scikit-learn pandas numpy tqdm pickle5 -q

In [ ]:
import os, re, pickle, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## Section 2 — Download Dataset from Kaggle

Upload your kaggle.json API key first (Kaggle → Account → Create New Token)

In [ ]:
# Upload kaggle.json
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

# Setup kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle credentials set!')

In [ ]:
# Download IMDB dataset
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip -q imdb-dataset-of-50k-movie-reviews.zip
!ls -lh *.csv

## Section 3 — DAG Node 1 & 2: Load & Clean Data

This is your data cleaning pipeline — find problems, fix them, report everything. Tell your teacher exactly what you found.

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────
df = pd.read_csv('IMDB Dataset.csv')
print(f'Raw dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nLabel distribution:')
print(df['sentiment'].value_counts())
print(f'\nSample review:')
print(df['review'].iloc[0][:200])

In [ ]:
# ── DAG Node 1: Data Cleaning ──────────────────────────────────────────
# This is real data science — explain every decision to your teacher

print('=== DATA CLEANING REPORT ===')
original_count = len(df)

issues = {
    'duplicates': 0,
    'empty_reviews': 0,
    'html_artifacts_fixed': 0,
    'very_short': 0,
    'very_long_truncated': 0,
}

# Check 1: Duplicates
dupes = df.duplicated(subset=['review']).sum()
issues['duplicates'] = dupes
df = df.drop_duplicates(subset=['review']).reset_index(drop=True)
print(f'Duplicates removed:        {dupes}')

# Check 2: Empty or null reviews
nulls = df['review'].isnull().sum()
empty = (df['review'].str.strip() == '').sum()
issues['empty_reviews'] = nulls + empty
df = df[df['review'].notna() & (df['review'].str.strip() != '')].reset_index(drop=True)
print(f'Empty/null reviews removed:{nulls + empty}')

# Check 3: HTML artifacts (fix, don't remove)
html_pattern = re.compile(r'<[^>]+>')
html_count = df['review'].str.contains(html_pattern).sum()
issues['html_artifacts_fixed'] = html_count
df['review'] = df['review'].apply(lambda x: re.sub(r'<[^>]+>', ' ', str(x)))
df['review'] = df['review'].apply(lambda x: re.sub(r'&amp;', '&', x))
df['review'] = df['review'].apply(lambda x: re.sub(r'&lt;', '<', x))
df['review'] = df['review'].apply(lambda x: re.sub(r'&gt;', '>', x))
df['review'] = df['review'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
print(f'HTML artifacts fixed:       {html_count} (not removed, cleaned in-place)')

# Check 4: Very short reviews (< 10 words — no useful signal)
word_counts = df['review'].str.split().str.len()
short_mask = word_counts < 10
issues['very_short'] = short_mask.sum()
df = df[~short_mask].reset_index(drop=True)
print(f'Very short reviews removed: {issues["very_short"]} (< 10 words)')

# Check 5: Very long reviews — truncate, not remove
MAX_WORDS = 500
long_mask = df['review'].str.split().str.len() > MAX_WORDS
issues['very_long_truncated'] = long_mask.sum()
df['review'] = df['review'].apply(
    lambda x: ' '.join(x.split()[:MAX_WORDS]) if len(x.split()) > MAX_WORDS else x
)
print(f'Long reviews truncated:     {issues["very_long_truncated"]} (capped at {MAX_WORDS} words)')

print(f'\nOriginal count:  {original_count}')
print(f'Final count:     {len(df)}')
print(f'Removed total:   {original_count - len(df)}')
print(f'\nFinal label distribution:')
print(df['sentiment'].value_counts())

# Save cleaned data
df.to_csv('imdb_clean.csv', index=False)
print('\nCleaned data saved to imdb_clean.csv')

## Section 4 — DAG Node 2: Text Preprocessing & Vocabulary

This is Week 11 NLP: tokenization, vocabulary building, text vectorization.

In [ ]:
# ── DAG Node 2: Text Preprocessing ────────────────────────────────────
# Week 11: NLP — Tokenization, stemming, text vectorization

def preprocess_text(text):
    """Clean and tokenize a review.
    Week 11 topics: tokenization, lowercasing, punctuation removal.
    """
    text = text.lower()                              # lowercase
    text = re.sub(r"[^a-z0-9\s']", ' ', text)       # remove special chars
    text = re.sub(r"\s+", ' ', text).strip()         # normalize whitespace
    tokens = text.split()                            # tokenize (split on space)
    return tokens

# Build vocabulary from training data
# Split first so vocab is built only on train set (no data leakage)
df['label'] = (df['sentiment'] == 'positive').astype(int)   # 1=positive, 0=negative

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['label'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Build vocabulary
print('\nBuilding vocabulary...')
word_counts = Counter()
for review in tqdm(train_df['review'], desc='Tokenizing'):
    tokens = preprocess_text(review)
    word_counts.update(tokens)

MIN_FREQ = 3          # words appearing < 3 times are noise
MAX_VOCAB = 30000     # cap vocabulary size

# Special tokens
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.most_common(MAX_VOCAB):
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print(f'Vocabulary size: {len(vocab):,} words')
print(f'Words with freq >= {MIN_FREQ}: {sum(1 for c in word_counts.values() if c >= MIN_FREQ):,}')
print(f'Total unique words: {len(word_counts):,}')

# Save vocabulary
with open('vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('Vocabulary saved to vocab.pkl')

## Section 5 — DAG Node 3 & 4: LSTM Model — Train From Scratch

Week 7-9: RNNs → LSTM. Forget gate, input gate, cell state, output gate — all working here.

In [ ]:
# ── Dataset class ──────────────────────────────────────────────────────
MAX_LEN = 256   # max tokens per review

class IMDBDataset(Dataset):
    def __init__(self, dataframe, vocab, max_len=MAX_LEN):
        self.reviews = dataframe['review'].tolist()
        self.labels  = dataframe['label'].tolist()
        self.vocab   = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = preprocess_text(text)[:self.max_len]
        ids = [self.vocab.get(t, 1) for t in tokens]   # 1 = <UNK>
        # Pad or truncate to max_len
        ids = ids + [0] * (self.max_len - len(ids))    # 0 = <PAD>
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        return self.encode(self.reviews[idx]), torch.tensor(self.labels[idx], dtype=torch.float)

train_ds = IMDBDataset(train_df, vocab)
val_ds   = IMDBDataset(val_df,   vocab)
test_ds  = IMDBDataset(test_df,  vocab)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)} | Test batches: {len(test_dl)}')

In [ ]:
# ── LSTM Model Definition ─────────────────────────────────────────────
# Week 7-9: This is the LSTM — explain forget gate, input gate, cell state, output gate

class SentimentLSTM(nn.Module):
    """
    LSTM-based sentiment classifier.
    Architecture:
      Embedding → BiLSTM (2 layers) → Attention pooling → FC → Sigmoid

    Week topics covered:
      - Week 4: Sigmoid activation in output, ReLU in FC
      - Week 7-9: LSTM gates (forget, input, cell, output)
      - Week 10: Hidden state = latent space representation
      - Week 12: Hidden state used for cosine similarity in RAG
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, n_layers=2, dropout=0.3):
        super().__init__()

        # Word Embedding: maps each token ID to a dense vector
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # BiLSTM: reads sequence forward AND backward
        # bidirectional=True → output is 2*hidden_dim
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0
        )

        # Dropout for regularization (Week 4)
        self.dropout = nn.Dropout(dropout)

        # Fully connected classifier head
        # hidden_dim*2 because bidirectional
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.fc2 = nn.Linear(128, 1)

        # Activations
        self.relu    = nn.ReLU()       # Week 4: ReLU in hidden layer
        self.sigmoid = nn.Sigmoid()    # Week 4: Sigmoid for binary output

    def forward(self, x, return_hidden=False):
        # x shape: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))          # (batch, seq_len, embed_dim)
        lstm_out, (hidden, cell) = self.lstm(embedded)      # lstm_out: (batch, seq_len, hidden*2)

        # Take the last hidden state from both directions
        # hidden: (n_layers*2, batch, hidden_dim)
        # Concat the last layer's forward & backward hidden states
        hidden_fwd = hidden[-2]   # last layer, forward direction
        hidden_bwd = hidden[-1]   # last layer, backward direction
        latent = torch.cat([hidden_fwd, hidden_bwd], dim=1)  # (batch, hidden*2)
        # ↑ This is the LATENT SPACE vector — Week 10 concept!
        # Like an autoencoder bottleneck, it compresses the review into 512 dims

        out = self.dropout(latent)
        out = self.relu(self.fc1(out))   # Week 4: ReLU activation
        out = self.dropout(out)
        logit = self.fc2(out)            # raw score
        prob  = self.sigmoid(logit)      # Week 4: Sigmoid → probability 0-1

        if return_hidden:
            return prob.squeeze(1), latent    # return latent for RAG
        return prob.squeeze(1)


VOCAB_SIZE  = len(vocab)
EMBED_DIM   = 128
HIDDEN_DIM  = 256
N_LAYERS    = 2
DROPOUT     = 0.3

model = SentimentLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'\nModel architecture:')
print(model)

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────
EPOCHS   = 10
LR       = 1e-3

criterion = nn.BCELoss()                          # Binary Cross-Entropy — Week 2
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

history   = []
best_val_acc = 0.0

def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x)
            loss  = criterion(preds, y)
            total_loss += loss.item()
            correct    += ((preds >= 0.5).float() == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset) * 100

print('Starting training...\n')
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct = 0, 0
    t0 = time.time()

    for x, y in tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()                    # clear gradients
        preds = model(x)                         # forward pass
        loss  = criterion(preds, y)              # compute loss (Week 2)
        loss.backward()                          # backpropagation (Week 3)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # gradient clipping
        optimizer.step()                         # update weights
        train_loss    += loss.item()
        train_correct += ((preds >= 0.5).float() == y).sum().item()

    train_loss /= len(train_dl)
    train_acc   = train_correct / len(train_dl.dataset) * 100
    val_loss, val_acc = evaluate(model, val_dl)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f'Epoch {epoch:2d}/{EPOCHS} │ '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% │ '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.1f}% │ '
          f'{elapsed:.0f}s')

    history.append({
        'epoch': epoch, 'train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4), 'train_acc': round(train_acc, 2),
        'val_acc': round(val_acc, 2)
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'lstm_model.pt')
        print(f'  => Saved best model (val_acc={val_acc:.1f}%)')

with open('training_history.json', 'w') as f:
    json.dump(history, f)

print(f'\nTraining complete! Best val accuracy: {best_val_acc:.1f}%')

## Section 6 — Test Set Evaluation

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load('lstm_model.pt', map_location=DEVICE))
test_loss, test_acc = evaluate(model, test_dl)

print('=== TEST SET RESULTS ===')
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.2f}%')
print(f'Test Samples:  {len(test_ds)}')

# Per-class accuracy
model.eval()
pos_correct, pos_total = 0, 0
neg_correct, neg_total = 0, 0

with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = (model(x) >= 0.5).float()
        for pred, label in zip(preds, y):
            if label == 1:
                pos_total += 1
                if pred == label: pos_correct += 1
            else:
                neg_total += 1
                if pred == label: neg_correct += 1

print(f'\nPositive accuracy: {pos_correct/pos_total*100:.1f}% ({pos_correct}/{pos_total})')
print(f'Negative accuracy: {neg_correct/neg_total*100:.1f}% ({neg_correct}/{neg_total})')

results = {'test_loss': round(test_loss,4), 'test_acc': round(test_acc,2), 'test_samples': len(test_ds)}
with open('eval_results.json','w') as f:
    json.dump(results, f)
print('\nSaved: eval_results.json')

## Section 7 — DAG Node 6: Build RAG Store

Week 10 + 12: Extract latent vectors (like autoencoder bottleneck) and save for cosine similarity search.

In [ ]:
# ── Build RAG Store ────────────────────────────────────────────────────
# Use 1000 samples from test set as reference corpus
# Each sample's latent vector (512-dim) will be searched via cosine similarity

model.load_state_dict(torch.load('lstm_model.pt', map_location=DEVICE))
model.eval()

RAG_SAMPLES = 1000
rag_df = test_df.sample(n=RAG_SAMPLES, random_state=42).reset_index(drop=True)

vectors  = []
texts    = []
labels   = []
snippets = []

rag_ds = IMDBDataset(rag_df, vocab)
rag_dl = DataLoader(rag_ds, batch_size=32, shuffle=False)

print('Extracting latent vectors for RAG store...')
with torch.no_grad():
    for x, y in tqdm(rag_dl, desc='RAG encoding'):
        x = x.to(DEVICE)
        _, latent = model(x, return_hidden=True)  # 512-dim latent vector
        vectors.append(latent.cpu().numpy())

vectors = np.vstack(vectors)   # shape: (1000, 512)

for i, row in rag_df.iterrows():
    texts.append(row['review'])
    labels.append(row['sentiment'])
    # Create a clean snippet (first 150 chars)
    clean = row['review'].strip()
    snippets.append(clean[:150] + '...' if len(clean) > 150 else clean)

rag_store = {
    'vectors':  vectors,     # (1000, 512) numpy array
    'texts':    texts,       # full review texts
    'labels':   labels,      # 'positive' or 'negative'
    'snippets': snippets,    # short previews for UI display
}

with open('rag_store.pkl', 'wb') as f:
    pickle.dump(rag_store, f)

print(f'\nRAG store saved!')
print(f'Vectors shape: {vectors.shape}')
print(f'Positive samples: {labels.count("positive")}')
print(f'Negative samples: {labels.count("negative")}')

## Section 8 — Quick Test: Predict a Review

In [ ]:
# Quick sanity check — predict a sample review
def predict_review(text, model, vocab, device=DEVICE):
    model.eval()
    tokens = preprocess_text(text)[:MAX_LEN]
    ids    = [vocab.get(t, 1) for t in tokens]
    ids    = ids + [0] * (MAX_LEN - len(ids))
    x      = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        prob, latent = model(x, return_hidden=True)
    prob = prob.item()
    sentiment = 'positive' if prob >= 0.5 else 'negative'
    return {'sentiment': sentiment, 'confidence': round(prob * 100 if prob >= 0.5 else (1-prob)*100, 1)}

test_reviews = [
    "This movie was absolutely brilliant! The acting was superb and the story kept me engaged throughout.",
    "What a waste of time. The plot made no sense and the characters were completely unlikeable.",
    "It was okay, nothing special. Some good moments but mostly forgettable."
]

print('=== QUICK PREDICTION TEST ===')
for review in test_reviews:
    result = predict_review(review, model, vocab)
    print(f'Review: "{review[:60]}..."')
    print(f'Prediction: {result["sentiment"]} ({result["confidence"]}% confident)\n')

## Section 9 — Download All Model Files

In [ ]:
# Zip and download all required files for the backend
import zipfile

files_to_save = [
    'lstm_model.pt',          # Trained LSTM weights
    'vocab.pkl',              # Vocabulary dictionary
    'rag_store.pkl',          # RAG reference vectors
    'training_history.json',  # Training curves
    'eval_results.json',      # Test accuracy
]

with zipfile.ZipFile('sentimentsage_models.zip', 'w') as zf:
    for f in files_to_save:
        if os.path.exists(f):
            zf.write(f)
            print(f'  Added: {f} ({os.path.getsize(f)/1024:.1f} KB)')

print('\nDownloading sentimentsage_models.zip...')
from google.colab import files
files.download('sentimentsage_models.zip')